In [18]:
!pip install pandas scikit-learn numpy joblib

In [33]:
# ==============================
# PROFESSIONAL SPAM DETECTION SYSTEM
# Hybrid ML + Rule-Based Intelligence
# ==============================

import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack
import csv # Import the csv module for QUOTE_NONE

# -----------------------------
# 1. LOAD DATASET (Your Path)
# -----------------------------

data_path = "spam.csv" # Changed path to reference the file in the current working directory

# Use Python engine and handle quoting, explicitly define names to manage varying column counts
df = pd.read_csv(data_path, sep='\t', names=['v1', 'v2', 'col3', 'col4', 'col5', 'col6', 'col7', 'col8', 'col9', 'col10'], encoding='latin-1', engine='python', quoting=csv.QUOTE_NONE)

# Select only the relevant 'v1' and 'v2' columns and rename them
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# Fill NaN values in the 'message' column with empty strings
df['message'] = df['message'].fillna('')

# Clean and convert labels: strip whitespace, lowercase, map to 0/1, then drop NaNs and convert to int
df['label'] = df['label'].str.strip().str.lower().map({'ham': 0, 'spam': 1})
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int)

print("Dataset Loaded Successfully")
print(df.head())

# -----------------------------
# 2. TEXT CLEANING
# -----------------------------

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['cleaned'] = df['message'].apply(clean_text)

# -----------------------------
# 3. FULL SPAM KEYWORD LIST
# -----------------------------

spam_keywords = [

    # Financial Spam
    "free money","guaranteed income","risk free","double your income",
    "earn","million dollars","act now","limited time offer",
    "cash bonus","winner","congratulations","claim your prize",
    "urgent response needed",

    # Phishing
    "verify your account","update payment details",
    "suspicious activity","account suspended",
    "reset your password","security alert",
    "login immediately","confirm identity",

    # Marketing
    "100 free","exclusive deal","best price","cheap",
    "discount","offer expires","buy now",
    "clearance sale","no hidden fees","lowest price",

    # Health
    "miracle cure","lose weight fast","no prescription",
    "instant results","secret formula",
    "anti aging","cure all",

    # Pressure
    "dont miss out","last chance","urgent",
    "only today","once in a lifetime",
    "hurry","immediate action",

    # Basic triggers
    "win","won","lottery","prize","claim",
    "free","click","offer","cash","reward"
]

# -----------------------------
# 4. KEYWORD FEATURE
# -----------------------------

def keyword_feature(text):
    text = text.lower()
    for word in spam_keywords:
        if word in text:
            return 1
    return 0

df['keyword_flag'] = df['message'].apply(keyword_feature)

# -----------------------------
# 5. FORMATTING FEATURES
# -----------------------------

def formatting_features(text):
    features = []

    text = str(text)

    features.append(int(text.isupper()))  # ALL CAPS
    features.append(int(text.count("!") > 3))  # Excessive !!!
    features.append(int(len(re.findall(r'[💰🔥🎉]', text)) > 1))  # Emojis
    features.append(int(len(re.findall(r'[$€£]', text)) > 0))  # Currency

    return np.array(features)

format_features = np.array(
    [formatting_features(text) for text in df['message']]
)

# -----------------------------
# 6. TF-IDF FEATURES
# -----------------------------

word_vectorizer = TfidfVectorizer(
    ngram_range=(1,3),
    max_features=5000
)

char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    max_features=3000
)

word_features = word_vectorizer.fit_transform(df['cleaned'])
char_features = char_vectorizer.fit_transform(df['cleaned'])

extra_features = np.column_stack([
    df['keyword_flag'],
    format_features
])

# Combine everything
X = hstack([word_features, char_features, extra_features])
y = df['label']

# -----------------------------
# 7. TRAIN MODEL
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# -----------------------------
# 8. EVALUATION
# -----------------------------

pred = model.predict(X_test)

print("\nModel Performance")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

# -----------------------------
# 9. SAVE MODEL TO DRIVE
# -----------------------------

joblib.dump(model, "/content/drive/MyDrive/spam_model.pkl")
joblib.dump(word_vectorizer, "/content/drive/MyDrive/word_vectorizer.pkl")
joblib.dump(char_vectorizer, "/content/drive/MyDrive/char_vectorizer.pkl")

print("\nModel Saved to Google Drive Successfully!")

Dataset Loaded Successfully
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...

Model Performance
Accuracy: 0.9820627802690582
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       954
           1       0.99      0.89      0.93       161

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115


Model Saved to Google Drive Successfully!


In [34]:
# -----------------------------
# REAL-TIME PREDICTION FUNCTION
# -----------------------------

def predict_message(text):

    cleaned = clean_text(text)

    # TF-IDF Features
    word_feat = word_vectorizer.transform([cleaned])
    char_feat = char_vectorizer.transform([cleaned])

    # Keyword feature
    keyword_flag = keyword_feature(text)

    # Formatting features
    format_feat = formatting_features(text).reshape(1, -1)

    extra = np.column_stack([ [keyword_flag], format_feat ])

    from scipy.sparse import hstack
    final_features = hstack([word_feat, char_feat, extra])

    # Prediction
    prediction = model.predict(final_features)[0]
    probability = model.predict_proba(final_features)[0][1]

    if prediction == 1:
        result = "SPAM"
    else:
        result = "HAM"

    print("\nMessage:", text)
    print("Prediction:", result)
    print("Spam Probability: {:.2f}%".format(probability*100))

    return result, probability

In [35]:
# Financial spam
predict_message("Congratulations! You have won 5 million dollars. Claim now!!!")

# Phishing
predict_message("Your account has been suspended. Verify immediately.")

# Marketing
predict_message("Exclusive deal! Buy now and get 70% discount.")

# Normal message
predict_message("Hey, are we meeting tomorrow for lunch?")

# Suspicious formatting
predict_message("FREE MONEY!!!!! 💰💰 CLICK NOW!!!")


Message: Congratulations! You have won 5 million dollars. Claim now!!!
Prediction: SPAM
Spam Probability: 75.62%

Message: Your account has been suspended. Verify immediately.
Prediction: HAM
Spam Probability: 9.57%

Message: Exclusive deal! Buy now and get 70% discount.
Prediction: HAM
Spam Probability: 28.24%

Message: Hey, are we meeting tomorrow for lunch?
Prediction: HAM
Spam Probability: 1.83%

Message: FREE MONEY!!!!! 💰💰 CLICK NOW!!!
Prediction: HAM
Spam Probability: 29.88%


('HAM', np.float64(0.2987741707693012))

In [36]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
print("Cross Validation Accuracy:", scores.mean())

Cross Validation Accuracy: 0.9793688159663798
